[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/05_normalization.ipynb)

# 05. Normalization

정규화 축과 통계량이 달라질 때 tensor가 어떻게 변하고 어떤 reduction이 생기는지 비교한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


device: cuda
torch: 2.11.0+cu128


In [2]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. BatchNorm2d

batch/channel 축 통계를 사용한다.


In [3]:
x = torch.tensor(
    [[[[1., 2.], [3., 4.]], [[2., 4.], [6., 8.]]],
     [[[2., 3.], [4., 5.]], [[1., 3.], [5., 7.]]]],
    device=device,
)
bn = nn.BatchNorm2d(2, affine=False, track_running_stats=False).to(device)
y = bn(x)
print(y)


tensor([[[[-1.6330, -0.8165],
          [ 0.0000,  0.8165]],

         [[-1.0911, -0.2182],
          [ 0.6547,  1.5275]]],


        [[[-0.8165,  0.0000],
          [ 0.8165,  1.6330]],

         [[-1.5275, -0.6547],
          [ 0.2182,  1.0911]]]], device='cuda:0')


In [4]:
_ = profile_call("BatchNorm2d", bn, x)



[BatchNorm2d] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                aten::native_batch_norm        19.43%       1.112ms        56.97%       3.259ms       3.259ms      14.016us       100.00%      28.032us      28.032us           0 B           0

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


## 2. InstanceNorm → GroupNorm

통계를 sample별 또는 channel group별로 제한한다.


In [5]:
inn = nn.InstanceNorm2d(2, affine=False, track_running_stats=False).to(device)
gn = nn.GroupNorm(1, 2, affine=False).to(device)

print("InstanceNorm:\n", inn(x))
print("GroupNorm:\n", gn(x))


InstanceNorm:
 tensor([[[[-1.3416, -0.4472],
          [ 0.4472,  1.3416]],

         [[-1.3416, -0.4472],
          [ 0.4472,  1.3416]]],


        [[[-1.3416, -0.4472],
          [ 0.4472,  1.3416]],

         [[-1.3416, -0.4472],
          [ 0.4472,  1.3416]]]], device='cuda:0')
GroupNorm:
 tensor([[[[-1.2702, -0.8083],
          [-0.3464,  0.1155]],

         [[-0.8083,  0.1155],
          [ 1.0392,  1.9630]]],


        [[[-0.9802, -0.4201],
          [ 0.1400,  0.7001]],

         [[-1.5403, -0.4201],
          [ 0.7001,  1.8204]]]], device='cuda:0')


In [6]:
_ = profile_call("InstanceNorm", inn, x)
_ = profile_call("GroupNorm", gn, x)



[InstanceNorm] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                aten::native_batch_norm         3.38%     109.794us        97.21%       3.162ms       3.162ms      13.184us       100.00%      26.368us      26.368us           0 B           

## 3. LayerNorm

마지막 feature 축을 정규화한다.


In [7]:
z = torch.tensor([[1., 2., 3., 4.], [2., 4., 6., 8.]], device=device)
ln = nn.LayerNorm(4, elementwise_affine=False).to(device)
print(ln(z))


tensor([[-1.3416, -0.4472,  0.4472,  1.3416],
        [-1.3416, -0.4472,  0.4472,  1.3416]], device='cuda:0')


In [8]:
_ = profile_call("LayerNorm", ln, z)



[LayerNorm] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                aten::native_layer_norm         1.91%      65.867us        99.33%       3.433ms       3.433ms       7.456us       100.00%      14.912us      14.912us           0 B           0 B

## 4. RMSNorm

평균을 빼지 않고 RMS로 scale만 맞춘다.


In [9]:
rms = nn.RMSNorm(4, elementwise_affine=False).to(device)
print(rms(z))

rms_explicit = z / torch.sqrt(z.pow(2).mean(dim=-1, keepdim=True) + 1e-5)
print("explicit close:", torch.allclose(rms(z), rms_explicit, atol=1e-4))


tensor([[0.3651, 0.7303, 1.0954, 1.4606],
        [0.3651, 0.7303, 1.0954, 1.4606]], device='cuda:0')
explicit close: True


In [10]:
_ = profile_call("RMSNorm", rms, z)



[RMSNorm] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  aten::_fused_rms_norm         2.19%      79.342us        98.72%       3.575ms       3.575ms       5.248us       100.00%      10.496us      10.496us           0 B           0 B  

## 5. QK-Norm

attention의 Q/K를 head dimension에서 따로 normalize한다.


In [11]:
q = torch.randn(1, 2, 4, 4, device=device)
k = torch.randn_like(q)

q_norm = F.rms_norm(q, (q.size(-1),))
k_norm = F.rms_norm(k, (k.size(-1),))

print("q RMS before:", q.pow(2).mean(-1).sqrt())
print("q RMS after :", q_norm.pow(2).mean(-1).sqrt())


q RMS before: tensor([[[0.6668, 0.9429, 0.5607, 0.7677],
         [1.1684, 1.0386, 1.5155, 0.4685]]], device='cuda:0')
q RMS after : tensor([[[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]]], device='cuda:0')


In [12]:
_ = profile_call("QK RMSNorm", lambda p, q_: (F.rms_norm(p, (p.size(-1),)), F.rms_norm(q_, (q_.size(-1),))), q, k)



[QK RMSNorm] top operators
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  aten::_fused_rms_norm         2.88%     104.753us        98.96%       3.594ms       1.797ms       9.984us       100.00%      15.104us       7.552us           0 B           0 

## References and provenance

**[5.1] BatchNorm**
- 출처: Ioffe & Szegedy, Batch Normalization
- 이 노트북에서 가져온 부분: batch/channel statistics

**[5.2] GroupNorm**
- 출처: Wu & He, Group Normalization
- 이 노트북에서 가져온 부분: channel grouping

**[5.3] LayerNorm**
- 출처: Ba et al., Layer Normalization
- 이 노트북에서 가져온 부분: feature-axis normalization

**[5.4] RMSNorm**
- 출처: Zhang & Sennrich, Root Mean Square Layer Normalization
- 이 노트북에서 가져온 부분: RMS-only normalization

**[5.5] QK-Norm**
- 출처: modern ViT/DiT/LLM implementations including Krea-family reports
- 이 노트북에서 가져온 부분: Q/K normalization before attention
